# HW 2, Problem 3: Build a square wave out of sines

A **square wave** has sharp corners, and a **sine wave** is the smoothest curve there is. Remarkably, if you add up enough sine waves, of the right frequencies and amplitudes, you can build the square wave anyway. This is the idea behind **Fourier series**, and it is how audio compression, radio receivers and the FFT (Fast Fourier Transform) work.

The linear algebra involved is all Unit 2 content: a signal sampled at $N$ times is a **vector**. Each sine and cosine is also a vector. The **inner product** with a sine measures how much of that frequency the signal contains, and stacking all the sines and cosines as the columns of a **matrix** $B$ lets two matrix products analyze and rebuild any signal:

$$\text{coefficients} = \tfrac{2}{N}\,B^T\,\text{signal}, \qquad \text{rebuilt signal} = B\,\text{coefficients}.$$

**ECE Connections:** Oh boy! Fourier transforms are at the absolute heart of a lot of what EE's and CE's do! The idea is to convert a signal over time (like an audio recording) into a list of magnitudes and phases and sinusoids, to work mathematically (e.g., via filters) directly in the so-called "frequency domain", with that sine wave info. Noise cancelling headphone technology; modulation and demodulation of AM and FM radio signals; 4G, 5G and Wifi signal handling; filter design for circuits; harmonic analysis of dangerous mechanical vibration and noise signals; analysis and spike prevention in power grids; radar, sonar and medical imaging -- and lots and lots more -- all involve frequency analysis (and Fourier decomposition).

<br>

**How to use this notebook:** run each of the cells, top to bottom, via the Run button or with **Shift + Enter**. Fill in every `...`, and write your guess at each **Predict first** before you run the code. Expect to take about 30 to 40 minutes. Try to break things in Part 5!

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

N = 1000                                   # samples in ONE period of the signal
t = np.arange(N) / N                       # times 0, 1/N, ..., just under 1 (the endpoint is left out on purpose)

square = np.where(t < 0.5, 1.0, -1.0)      # +1 for the first half period, -1 for the second half
square[[0, N // 2]] = 0.0                  # at each jump, use the midpoint value 0
tri = np.where(t < 0.5, 4*t - 1, 3 - 4*t)  # a triangle wave: from -1 up to +1 and back down

def grid_plot(signal, Ks, fit):
    '''Plot the signal (gray) and its rebuilt version (red) using K harmonics, for each K in Ks.
    `fit` is your fourier_fit function.'''
    fig, axes = plt.subplots(2, 3, figsize=(11, 5), sharex=True, sharey=True)
    for ax, K in zip(axes.ravel(), Ks):
        coeffs, rebuilt = fit(signal, K)
        ax.plot(t, signal, color="lightgray", lw=3)
        ax.plot(t, rebuilt, color="tab:red", lw=1.2)
        ax.set_title(f"K = {K} harmonics"); ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

def loglog_plot(xs, curves, ylabel, title):
    plt.figure(figsize=(6, 4))
    for label, ys in curves.items():
        plt.loglog(xs, np.maximum(ys, 1e-17), "o-", label=label)
    plt.xlabel("K (number of harmonics)"); plt.ylabel(ylabel)
    plt.title(title); plt.grid(True, which="both", alpha=0.3); plt.legend(); plt.show()

def check(ok, hint):
    print("OK - looks right!" if ok else "NOT YET: " + hint)

plt.figure(figsize=(8, 3))
plt.plot(t, square, label="square wave"); plt.plot(t, tri, label="triangle wave")
plt.xlabel("time (one period)"); plt.grid(True); plt.legend(); plt.show()
print("each signal is a vector with", square.shape[0], "entries")

# Part 1: How much sine is in a square wave?

A sine with $k$ cycles per period is the vector $\sin(2\pi k t)$. Its inner product with the square wave tells us how much of that frequency the square wave contains.

**Predict first:** the square wave looks a lot like one sine cycle. Which of these will have a **big** inner product with it, and which will be **close to zero**? $\sin(2\pi t)$, $\sin(4\pi t)$ (2 cycles), $\sin(6\pi t)$ (3 cycles), $\cos(2\pi t)$.

**Your prediction:**

## Task 1.1: Inner products with sines and cosines

In [ ]:
for k in [1, 2, 3]:
    s_k = np.sin(2*np.pi*k*t)            # a sine with k cycles per period
    c_k = ...                            # the matching cosine
    print(f"k={k}:  square . sin = {np.inner(square, s_k):8.2f}    square . cos = {np.inner(square, c_k):8.2f}")

**Question 1:** Which frequencies are present in a square wave, and which are missing (inner product 0)? Is there a pattern in which $k$ are present? Are cosines needed at all?

**Your answer:**

## Task 1.2: Turning an inner product into a coefficient
A sine of size 1 has inner product $N/2$ with itself (you will check this in Part 2). So if the square wave contains an amount $b_1$ of $\sin(2\pi t)$, then $b_1 = \frac{2}{N}\times$ (the inner product). Compute $b_1$ and compare it with $4/\pi \approx 1.273$.

In [ ]:
s1 = np.sin(2*np.pi*t)
b1 = ...                         # (2/N) times the inner product of the square wave with s1
print(b1, "   vs   4/pi =", 4/np.pi)
check(abs(b1 - 4/np.pi) < 0.001, "b1 should be about 1.273")

# Part 2: Stack them into a matrix

## Task 2.1: The basis matrix
Let `make_basis(K)` return an $N \times 2K$ matrix `B`. Its first $K$ columns are the cosines $\cos(2\pi k t)$ for $k = 1, \dots, K$ and its last $K$ columns are the sines $\sin(2\pi k t)$. (`np.outer(t, ks)` is a handy matrix whose entry in row $n$, column $k$ is $t_n \cdot k$.)

In [ ]:
def make_basis(K):
    ks = np.arange(1, K + 1)                       # frequencies 1, 2, ..., K  (cycles per period)
    cos_cols = np.cos(2*np.pi*np.outer(t, ks))     # N x K: column k-1 is cos(2 pi k t)
    sin_cols = ...                                 # N x K: the same for sine
    return np.hstack([cos_cols, sin_cols])         # N x 2K

B = make_basis(5)
print(B.shape)
check(B.shape == (N, 10) and np.allclose(B[:, 5 + 2], np.sin(2*np.pi*3*t)), "column K+2 (here 7) should be sin(2 pi 3 t)")

## Task 2.2: Are they orthogonal?
$G = B^T B$ contains **every inner product between every pair of columns** of $B$ (entry $i, j$ is column $i$ dotted with column $j$).

**Predict first:** what do you expect on the diagonal? Off the diagonal? (Remember what an inner product of $0$ means for two vectors.)

**Your prediction:**

In [ ]:
G = ...                          # B transposed times B
print(np.round(G, 3)[:4, :4])    # the top-left corner
check(np.allclose(G, (N/2) * np.eye(10)), "G should be (N/2) times the identity matrix")

**Question 2:** What do the zeros off the diagonal say about any two different sines and cosines? What is the value on the diagonal? Use these two facts to explain why $\frac{2}{N} B^T x$ gives the coefficients (**hint**: if $x = B c$, then $B^T x = B^T B\, c = \frac{N}{2}\, c$).

**Your answer:**

# Part 3: Analyze, then rebuild

## Task 3.1: `fourier_fit`
Write `fourier_fit(signal, K)`. It should compute **all** the $2K$ coefficients at once, with one matrix-vector product, and then rebuild the signal from them with another.

In [ ]:
def fourier_fit(signal, K):
    B = make_basis(K)
    coeffs = ...                 # all 2K coefficients: (2/N) times B.T @ signal
    rebuilt = ...                # the rebuilt signal: B @ coeffs
    return coeffs, rebuilt

coeffs, rebuilt = fourier_fit(square, 25)
for k in range(1, 8):
    print(f"k={k}:  cos coeff {coeffs[k-1]:7.3f}    sin coeff {coeffs[25 + k - 1]:7.3f}    (4/(pi k) = {4/(np.pi*k):.3f})")
check(abs(coeffs[25] - 4/np.pi) < 1e-3 and np.allclose(rebuilt, make_basis(25) @ coeffs), "the sin coefficient for k=1 should be about 1.273")
plt.figure(figsize=(8, 3)); plt.plot(t, square, color="lightgray", lw=3, label="square wave")
plt.plot(t, rebuilt, color="tab:red", label="rebuilt from 25 harmonics"); plt.legend(); plt.grid(True); plt.show()

**Question 3:** Which coefficients are (essentially) zero, and which are not? Write the formula for the square wave: $\text{square}(t) \approx \sum_{k\ \text{odd}} \_\_\_\ \sin(2\pi k t)$.

**Your answer:**

# Part 4: How many harmonics are enough?

## Task 4.1: Watch it build
**Predict first:** what does the rebuilt wave look like with just $K = 1$? With $K = 3$?

**Your prediction:**

In [ ]:
grid_plot(square, [1, 3, 5, 9, 25, 99], fourier_fit)

In [ ]:
for K in [25, 99]:
    _, rebuilt = fourier_fit(square, K)
    print(f"K = {K:3d}: the rebuilt wave peaks at {rebuilt.max():.3f}   (the square wave itself only reaches 1)")

**Question 4:** Describe what happens near the jumps as $K$ grows. Does the overshoot at the corners go away? About how tall is it compared with the size of the jump (from $-1$ to $+1$ is a jump of 2)? (This is called the **Gibbs phenomenon**.)

**Your answer:**

## Task 4.2: Error vs. number of harmonics
The **RMS error** is the norm of the difference divided by $\sqrt{N}$: a typical size of the mistake. Compare how fast the error falls for the square wave and for the triangle wave, on a log-log plot (like in the pi problem!).

In [ ]:
def rms_error(signal, K):
    _, rebuilt = fourier_fit(signal, K)
    return ...                   # the norm of (signal - rebuilt), divided by np.sqrt(N)

Ks = [1, 3, 5, 9, 15, 25, 50, 99]
err_square = [rms_error(square, K) for K in Ks]
err_tri    = [rms_error(tri, K) for K in Ks]
loglog_plot(Ks, {"square wave": err_square, "triangle wave": err_tri}, "RMS error", "Error vs. number of harmonics")
slope_sq  = np.polyfit(np.log10(Ks), np.log10(err_square), 1)[0]
slope_tri = np.polyfit(np.log10(Ks), np.log10(err_tri), 1)[0]
print(f"slope for the square wave: {slope_sq:.2f}      slope for the triangle wave: {slope_tri:.2f}")
check(abs(slope_sq + 0.5) < 0.1 and slope_tri < slope_sq - 0.5, "square slope about -0.5, triangle clearly steeper")

**Question 5:** Which wave is easier to build out of sines and cosines, and why do you think so? (Hint: look at the corners of each wave. The triangle's coefficients shrink like $1/k^2$ and the square's like $1/k$.)

**Your answer:**

# Part 5: Try to break it!

**Break it #1: too many frequencies.** Our $N = 1000$ samples can only tell $N/2 = 500$ frequencies apart. What happens to the orthogonality (the check from Task 2.2) when $K$ gets too big?

In [ ]:
for K in [250, 499, 500, 600]:
    B = make_basis(K)
    print(f"K = {K}: columns still orthogonal with norm^2 = N/2?  {np.allclose(B.T @ B, (N/2) * np.eye(2*K))}")

**Break it #1 question:** At what $K$ does it stop working? (Hint: what is the vector $\sin(2\pi\cdot 500\cdot t)$ at the times $t = n/1000$?) This limit is called the **Nyquist limit**: you can't see frequencies above half the sampling rate.

**Your answer:**

**Break it #2: shift the wave.** Delay the square wave by 37 samples with `np.roll`. **Predict first:** will the sine coefficients stay the same? Will the cosine coefficients stay zero?

In [ ]:
shifted = np.roll(square, 37)
c0, _ = fourier_fit(square, 25)
c1, _ = fourier_fit(shifted, 25)
print("            k=1      k=3      k=5")
print("original  sin:", np.round(c0[[25, 27, 29]], 3), "  cos:", np.round(c0[[0, 2, 4]], 3))
print("shifted   sin:", np.round(c1[[25, 27, 29]], 3), "  cos:", np.round(c1[[0, 2, 4]], 3))
print("sqrt(sin^2 + cos^2), original:", np.round(np.sqrt(c0[[25, 27, 29]]**2 + c0[[0, 2, 4]]**2), 3))
print("sqrt(sin^2 + cos^2), shifted: ", np.round(np.sqrt(c1[[25, 27, 29]]**2 + c1[[0, 2, 4]]**2), 3))

**Break it #2 question:** What changed and what stayed the same after the shift? What does that say about how much of each frequency is present versus *when* the wave starts?

**Your answer:**

**Break it #3: your own wave.** Make up any signal, a vector of $N$ numbers, and watch it get rebuilt. Subtract its average first (our basis has no constant column), e.g. `2*t - 1` is a sawtooth, or `np.where(t < 0.25, 1.0, 0.0)` is a pulse.

In [ ]:
my_signal = ...                  # your own wave: a vector with N entries
my_signal = my_signal - my_signal.mean()
grid_plot(my_signal, [1, 3, 5, 9, 25, 99], fourier_fit)

**Break it #3 question:** What wave did you make? Did it need many harmonics or few? Where did it have trouble, and why?

**Your answer:**

# What you found
- The square wave contains only **_____** (odd/even) frequencies, all of them **_____** (sines/cosines), with sizes $\_\_\_$.
- The overshoot at a jump is about **_____** (% of the jump) and does it go away as $K$ grows? **_____**
- Error slopes: square **_____**, triangle **_____**. Smoother waves need **_____** (more/fewer) harmonics.
- With $N$ samples, frequencies above **_____** cannot be told apart.

**Before you submit:** use **Kernel → Restart & Run All** and make sure nothing gives an error. Save and submit on Gradescope.